# Information Retrieval of Hospital Emergency Department Disruptions

This notebook implements the **discovery and refining stages** of the pipeline. First, targeted web searches are performed for each hospital using the Tavily Search API. The retrieved documents are then cleaned and enriched with metadata in order to prepare them for the LLM extraction stage.


The objective of this step is to collect online sources that may describe disruptions affecting hospital emergency departments (e.g., temporary closures, regulated access, or service reductions).

For each hospital in the dataset, we perform targeted web searches using the **Tavily Search API**, a search engine designed for retrieval tasks in language-model pipelines. Tavily returns relevant webpages together with metadata and extracted textual content.

Each search query combines:
- the hospital name
- alternative keywords identifying the hospital
- terms related to emergency department activity (e.g., *urgences*, *fermeture*, *régulation*)

The goal is to retrieve documents such as:
- press articles
- hospital announcements
- communications from regional health authorities
- local government notices
- institutional press releases

These retrieved documents will later be analyzed by a **Large Language Model (LLM)** to extract structured information about emergency department disruptions.

## Structure of the Retrieved Data

Each observation in the resulting dataset corresponds to **one retrieved source returned by the Tavily search engine**.

For each retrieved document, the following information is stored:

- **finess** – hospital identifier  
- **hospital_name** – short hospital name  
- **nom_etab_long** – full hospital name  
- **keywords_nom_etab** – alternative identifiers used in search queries  
- **source_url** – URL of the retrieved webpage  
- **title** – title of the retrieved page  
- **content** – extracted textual content of the page  
- **score** – Tavily relevance score  
- **retrieved_at** – timestamp of the retrieval  

The **content** field contains the raw textual material that will later be processed by the LLM extraction pipeline in order to identify and structure information about emergency department disruptions.

In [35]:
import json
import time
import os
import pandas as pd
from tavily import TavilyClient
import asyncio
from tavily import AsyncTavilyClient

# Initialize Tavily
tavily_client = TavilyClient(api_key="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") #4k
#tavily_client = TavilyClient(api_key="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") #1k

The following dataframe contains the cleaned reference table derived from the Enquête Urgences dataset. This table provides the list of hospital establishments that will be used as the starting point for the information retrieval process.

Each row corresponds to a hospital establishment with an emergency department.

In [24]:
df = pd.read_csv(
    "../data/intermediate/Emergency Room Lookup.csv",
    sep=";",          # ← this is the key
    encoding="utf-8",
)

df = df.rename(columns={'id': 'record_id'})
df["comm"] = df["comm"].astype("Int64")
df["siret"] = df["siret"].astype("Int64")
df = df.drop(columns=["Unnamed: 0"])
df.head()

,record_id,finess,nom_etab_1,nom_etab_2,nom_etab_long,compl_etab,compl_distr,achemin,comm,dept,...,siret,code_ape,code_mft,lib_mft,code_sph,lib_sph,date_ouv,date_autor,date_maj,num_educ
0,010000024G1,010000024,CH DE FLEYRIAT,CH DE FLEYRIAT,CENTRE HOSPITALIER DE BOURG-EN-BRESSE FLEYRIAT,NaN,NaN,01440 VIRIAT,451,01,...,26010004500012,8610Z,3.0,ARS établissements Publics de santé dotation g...,1.0,Etablissement public de santé,1979-02-13,1979-02-13,2020-02-04,NaN
1,010000032G1,010000032,CH BUGEY SUD,CH BUGEY SUD,CENTRE HOSPITALIER BUGEY SUD,NaN,NaN,01300 BELLEY,34,01,...,26010003700068,8610Z,3.0,ARS établissements Publics de santé dotation g...,1.0,Etablissement public de santé,1901-01-01,1901-01-01,2021-07-07,NaN
2,010005239G1,010005239,CH DU HAUT BUGEY - GEOVREISSET,CH DU HAUT BUGEY - GEOVREISSET,CENTRE HOSPITALIER DU HAUT BUGEY - GEOVREISSET,NaN,NaN,01108 OYONNAX CEDEX,283,01,...,26011021800112,NaN,3.0,ARS établissements Publics de santé dotation g...,1.0,Etablissement public de santé,2007-04-20,2007-03-14,2018-01-12,NaN
3,010780195G1,010780195,CLINIQUE CONVERT,CLINIQUE CONVERT,CLINIQUE DOCTEUR CONVERT,NaN,NaN,01000 BOURG EN BRESSE,53,01,...,77220148900022,8610Z,7.0,ARS établissements de santé non financés dotat...,0.0,Non concerné,1956-11-16,1956-11-16,2021-10-18,NaN
4,010780203G1,010780203,HOPITAL PRIVE D AMBERIEU,HOPITAL PRIVE D'AMBERIEU,HOPITAL PRIVE D'AMBERIEU,NaN,NaN,01506 AMBERIEU EN BUGEY CEDEX,4,01,...,81157144700010,8610Z,7.0,ARS établissements de santé non financés dotat...,0.0,Non concerné,2001-09-03,1994-04-03,2024-01-30,NaN


In [25]:
# Create a summaary to review the structure of the dataset
summary = pd.DataFrame({
    'Unique Values': df.nunique(),
    'Data Type': df.dtypes,
    'Missing Values (Total)': df.isnull().sum(),
    'Missing Values (%)': (df.isnull().sum() / len(df)) * 100
})
# Sort by percentage of missing values
summary = summary.sort_values(by='Missing Values (%)', ascending=False)

print(summary)

                Unique Values Data Type  Missing Values (Total)  \
num_educ                   13    object                     701   
compl_distr                34    object                     673   
compl_etab                 51    object                     641   
num_voie                  155   float64                     176   
nom_etab_long             510    object                     106   
code_ape                    4    object                      70   
typ_voie                   21    object                      18   
lib_voie                  533    object                      13   
siret                     599     Int64                       9   
lib_sph                     6    object                       2   
code_sph                    6   float64                       2   
achemin                   598    object                       1   
categ_etab_agr              5   float64                       1   
lib_etab                    5    object                       

In [26]:
df.shape

(715, 28)

In [27]:
df["nom_etab_1"].str.contains("CH").mean()

np.float64(0.5692307692307692)

## Cleaning Hospital Names

Administrative hospital names often include prefixes such as *Centre Hospitalier*, *Clinique*, or *Hôpital Privé*.  
These prefixes are frequently omitted in news articles and public announcements. While these prefixes are useful for administrative identification, they are frequently not used in news articles, public announcements, or online references. Instead, sources typically refer to hospitals using the specific site name or geographic identifier (for example: Fleyriat, Belley, etc.).

To improve the relevance of the web search queries, we construct a simplified keyword version of the hospital name by removing these administrative prefixes.

In [28]:
def clean_long_name(name):
    """
    Strips administrative prefixes from long hospital names 
    to extract the specific location or site name.
    """
    if not name or pd.isna(name):
        return ""
    
    # List of prefixes to remove, ordered from longest to shortest
    # to avoid partial replacement issues.
    prefixes = [
        "CENTRE HOSPITALIER REGIONAL ET UNIVERSITAIRE DE ",
        "CENTRE HOSPITALIER UNIVERSITAIRE DE ",
        "CENTRE HOSPITALIER REGIONAL DE ",
        "CENTRE HOSPITALIER DE ",
        "CENTRE HOSPITALIER ",
        "CH ",
        "HOPITAL PRIVE DE ",
        "HOPITAL PRIVE D' ",
        "HOPITAL PRIVE ",
        "CLINIQUE DOCTEUR ",
        "CLINIQUE DU ",
        "CLINIQUE DE ",
        "CLINIQUE "
    ]
    
    clean = str(name).upper()
    
    for p in prefixes:
        if clean.startswith(p):
            clean = clean.replace(p, "", 1) # Only replace the first occurrence
            break # Once a prefix is matched and removed, we stop
            
    return clean.strip()

# Apply to your dataframe
# Use nom_etab_long when available, fall back to nom_etab if NaN
df['keywords_nom_etab'] = df.apply(
    lambda x: clean_long_name(x['nom_etab_long']) if pd.notna(x['nom_etab_long']) 
    else clean_long_name(x['nom_etab_1']), 
    axis=1
)
df["keywords_nom_etab"].isna().sum()

np.int64(0)

In [29]:
df["ville"] = df["achemin"].str.replace(r"^\d{5}\s+", "", regex=True)
df_unique = df[["finess", "nom_etab_1", "nom_etab_long",'keywords_nom_etab', "ville", "record_id"]].reset_index(drop=True)
df_unique.head(10)

,finess,nom_etab_1,nom_etab_long,keywords_nom_etab,ville,record_id
0,010000024,CH DE FLEYRIAT,CENTRE HOSPITALIER DE BOURG-EN-BRESSE FLEYRIAT,BOURG-EN-BRESSE FLEYRIAT,VIRIAT,010000024G1
1,010000032,CH BUGEY SUD,CENTRE HOSPITALIER BUGEY SUD,BUGEY SUD,BELLEY,010000032G1
2,010005239,CH DU HAUT BUGEY - GEOVREISSET,CENTRE HOSPITALIER DU HAUT BUGEY - GEOVREISSET,DU HAUT BUGEY - GEOVREISSET,OYONNAX CEDEX,010005239G1
3,010780195,CLINIQUE CONVERT,CLINIQUE DOCTEUR CONVERT,CONVERT,BOURG EN BRESSE,010780195G1
4,010780203,HOPITAL PRIVE D AMBERIEU,HOPITAL PRIVE D'AMBERIEU,D'AMBERIEU,AMBERIEU EN BUGEY CEDEX,010780203G1
5,020000162,CH SAINT-QUENTIN,NaN,SAINT-QUENTIN,ST QUENTIN CEDEX,020000162A1
6,020000162,CH SAINT-QUENTIN,NaN,SAINT-QUENTIN,ST QUENTIN CEDEX,020000162P1
7,020000394,CH LAON,NaN,LAON,LAON CEDEX,020000394A1
8,020000394,CH LAON,NaN,LAON,LAON CEDEX,020000394P1
9,020000519,CH SOISSONS,NaN,SOISSONS,SOISSONS CEDEX,020000519A1


In [30]:
df_unique = df.drop_duplicates(subset="finess", keep="first")

In [31]:
print(len(df_unique["nom_etab_1"].unique().tolist()))

609


In [32]:
print(len(df_unique["finess"].unique().tolist()))

609


In [33]:
print(len(df_unique["keywords_nom_etab"].unique().tolist()))

603


The dataset contains **609 unique FINESS codes**, corresponding to **609 emergency department establishments**.  
Each row represents a hospital site with an emergency department that will be used as the starting point for the web search process.

## Query Construction

For each hospital in the reference table, we construct a targeted web search query using:
* the short hospital name (nom_etab_1)
* the cleaned keyword version of the hospital name (keywords_nom_etab)
* terms related to emergency department disruptions, such as fermeture temporaire, grève, SMUR, and communiqué de presse

The purpose of this query design is to improve retrieval of sources that may describe temporary closures, regulated access, staffing-related disruptions, or official hospital communications concerning emergency department operations.

The search is performed using the Tavily API with the following settings:
* advanced search depth to improve recall
* general topic rather than news, as the latter returned many irrelevant results
* exact match to better preserve hospital-specific retrieval
* raw content included so that the retrieved text can later be processed by the extraction pipeline
* a French geographic filter
* a restricted time window from January 2023 to December 2025

Each retrieved result is stored in atomic format, meaning that one row corresponds to one retrieved source.
This is important because different sources may describe different disruption events for the same hospital.

For each source, we save:
1. the hospital identifiers (finess, hospital_name, nom_etab_long)
2. the cleaned keyword used for retrieval
3. the source URL
4. the page title
5. the retrieved text content
6. the Tavily relevance score
7. the retrieval timestamp

The output is written incrementally to a JSONL file, which allows the search process to be monitored and resumed without having to rebuild the full dataset each time.


In [34]:
# 1. Setup
# Define the output jsonl file
output_path_simple = "/Data/anahi_reyes/EDCD_data/raw_edcd_database_atomic.jsonl"
folder_path = os.path.dirname(output_path_simple )

# Create folder if it does not exist
os.makedirs(folder_path, exist_ok=True)
print(f"Verified directory: {output_path_simple }")

Verified directory: /Data/anahi_reyes/EDCD_data/raw_edcd_database_atomic.jsonl


**Parameters Used**

- **`query`** *(dynamic string)*  
  The search query to execute. It combines the hospital name, keywords, and French emergency-closure terms.

- **`search_depth` = `"advanced"`**  
  Highest relevance with increased latency. This setting returns multiple semantically relevant snippets per URL (controlled by `chunks_per_source`). It costs **2 API credits per request**.

- **`topic` = `"general"`**  
  Broad, general-purpose search across a wide range of sources. Other options include `"news"` (real-time updates from mainstream media) and `"finance"`.

- **`chunks_per_source` = `3`**  
  Maximum number of relevant content snippets returned per source (each snippet can be up to ~500 characters).  
  This parameter is only available when `search_depth="advanced"`. The allowed range is **1–3**.

- **`max_results` = `10`**  
  Maximum number of search results returned per query. The allowed range is **0–20**, with a default value of **5**.

- **`country` = `"france"`**  
  Boosts search results originating from France. This parameter is only available when `topic="general"`.

- **`start_date` = `"2022-12-01"`**  
  Returns results that were published or updated **after this date**.  
  Format: `YYYY-MM-DD`.

- **`end_date` = `"2025-12-31"`**  
  Returns results that were published or updated **before this date**.  
  Format: `YYYY-MM-DD`.

**Commented-Out Parameters**

- **`include_raw_content`**  
  If enabled, includes cleaned and parsed HTML content of each result (returned as markdown or plain text). I did not used this approach because the LLM model often gets confused with more text.

- **`exact_match`**  
  If enabled, ensures that only results containing the **exact quoted phrase(s)** in the query are returned, bypassing synonyms or semantic variations. When using this approach, I get less relevant results. 

In [ ]:
for index, row in df_unique.iterrows(): # Use df_unique.head(2) to iterate with just two ED
    finess = row["finess"]
    name = row["nom_etab_1"]
    keywords_nom_etab = row["keywords_nom_etab"]
    nom_etab_long = row["nom_etab_long"]

    try:
        query = f'("{name}" OR "{keywords_nom_etab}") urgences ("fermeture temporaire" OR "accès régulé" OR "régulation des urgences" OR "grève" OR "effectif insuffisant" OR "fermeture nuit")'

        search = tavily_client.search(
            query=query,
            search_depth="advanced",
            topic="general", # "news"
            chunks_per_source=3,
            #include_raw_content=True,
            #exact_match=True,
            max_results=10,
            country="france",
            start_date="2022-12-01",
            end_date="2025-12-31"
        )  

        results = search.get('results', [])

        # Save ONE ROW PER SOURCE (atomic format)
        # Each source likely describes a different closure event
        for result in results:
            entry = {
                "finess": finess,
                "hospital_name": name,
                "nom_etab_long": nom_etab_long,
                "keywords_nom_etab": keywords_nom_etab,
                "source_url": result.get("url"),
                "title": result.get("title"),
                "content": result.get("content"),
                "score": result.get("score"),
                "retrieved_at": time.strftime("%Y-%m-%d %H:%M:%S")
            }

            with open(output_path_simple, "a", encoding="utf-8") as f:
                f.write(json.dumps(entry, ensure_ascii=False) + "\n")

        print(f"✅ {name}: {len(results)} sources found")
        time.sleep(1)

    except Exception as e:
        print(f"❌ Error with {name}: {str(e)}")


print(f"Deep context dataset saved in: {output_path_simple}")

✅ CH DE FLEYRIAT: 10 sources found
✅ CH BUGEY SUD: 10 sources found
✅ CH DU HAUT BUGEY - GEOVREISSET: 10 sources found
✅ CLINIQUE CONVERT: 10 sources found
✅ HOPITAL PRIVE D AMBERIEU: 10 sources found
✅ CH SAINT-QUENTIN: 10 sources found
✅ CH LAON: 10 sources found
✅ CH SOISSONS: 10 sources found
✅ CH CHAUNY: 10 sources found
✅ CH CHATEAU-THIERRY: 10 sources found
✅ CH HIRSON: 10 sources found
✅ HOPITAL PRIVE SAINT CLAUDE: 10 sources found
✅ CH DE MOULINS: 10 sources found
✅ CH DE MONTLUCON: 10 sources found
✅ CH JACQUES LACARIN VICHY: 10 sources found
✅ CHI DE MANOSQUE LOUIS RAFFALLI: 10 sources found
✅ CHI DES ALPES DU SUD SITE DE SISTERON: 10 sources found
✅ CENTRE HOSPITALIER DE DIGNE LES BAINS: 10 sources found
✅ CH DES ESCARTONS DE BRIANCON: 10 sources found
✅ CENTRE HOSPITALIER D EMBRUN: 10 sources found
✅ CHI DES ALPES DU SUD SITE DE GAP: 10 sources found
✅ CENTRE HOSPITALIER DE GRASSE: 10 sources found
✅ CH D ANTIBES JUAN LES PINS: 10 sources found
✅ CH DE CANNES SIMONE VEIL: 

### Attempt 1 — Dual Search Streams (Official vs Unofficial Sources)

This initial approach attempted to retrieve information about emergency department disruptions using **two separate search streams**:

1. **Official sources** (government and administrative websites)
2. **Unofficial sources** (news outlets, press releases, and general web content)

The goal of this design was to separate **institutional announcements** from **media reporting**, under the assumption that official sources would provide the most reliable information regarding hospital service disruptions.

### Official Stream

The official stream restricted the search to specific government domains using `site:` operators:

- `ars.sante.fr`
- `prefectures-regions.gouv.fr`
- `sante.gouv.fr`

These domains correspond to:

- **Regional Health Agencies (ARS)**
- **Prefectures**
- **French Ministry of Health**

The query combined:

- hospital identifiers (`name` or `clean_key`)
- emergency department keywords (`urgences`)
- disruption-related terms (`fermeture`, `régulation`, `accès régulé`)
- strict domain filters

The objective was to retrieve **administrative decisions or official communications** about emergency department closures or regulated access.

### Unofficial Stream

The second stream searched the **open web without domain restrictions**.

The query included:

- hospital identifiers
- emergency department terms
- disruption-related expressions such as  
  *fermeture temporaire*, *grève*, *SMUR*, and *communiqué de presse*
- the keyword **France** to avoid international results
- a time hint (`2022..2026`) to bias the search toward recent events.

This stream aimed to capture **news coverage, local press articles, and hospital communications** that may not appear on official administrative websites.

### Data Structure

For each hospital, the pipeline stored the results of both searches in a single JSON entry:

- hospital identifiers (`record_id`, `finess`, `hospital_name`)
- results from the **official search**
- results from the **unofficial search**
- the retrieval timestamp

The results were saved in **JSONL format**, with one entry per hospital.

### Why This Approach Was Abandoned

Although conceptually appealing, this design introduced several practical issues:

- **Official sources rarely contained the relevant announcements**, as many disruptions are communicated through hospital websites or local media rather than national or regional administration pages.
- The separation between **official and unofficial streams increased pipeline complexity** without substantially improving information retrieval.
- In practice, **most relevant evidence was found through general web searches**, making the strict domain filtering unnecessary.

For these reasons, the retrieval pipeline was later simplified to a **single targeted query strategy**, focusing on maximizing recall while preserving hospital-specific relevance.

This earlier implementation is preserved here for documentation purposes.

In [ ]:
# 1. Setup
# Define the output jsonl file
output_path = "/Data/anahi_reyes/EDCD_data/raw_edcd_database.jsonl"
folder_path = os.path.dirname(output_path)

# Create folder if it does not exist
os.makedirs(folder_path, exist_ok=True)
print(f"Verified directory: {output_path}")

Verified directory: /Data/anahi_reyes/EDCD_data/raw_edcd_database.jsonl


In [121]:
# We keep Official domains strict to ensure "Legal Truth"
official_domains = "site:ars.sante.fr OR site:prefectures-regions.gouv.fr OR site:sante.gouv.fr"

# 2. Main Loop
for index, row in df.iterrows():
    name = row["nom_etab"]
    clean_key = row["clean_key"]
    finess = row["finess"]
    record_id = row["record_id"]
    
    # --- STREAM 1: OFFICIAL (Strict) ---
    query_official = (
        f'("{name}" OR "{clean_key}") '
        f'(urgences OR "service des urgences") '
        f'(fermeture OR "régulation" OR "accès régulé") '
        f'({official_domains})'
    )

    # --- STREAM 2: UNOFFICIAL (Open & Global) ---
    # No "site:" restrictions here. We add "communique de presse" and "France" to avoid generic international results.
    query_unofficial = (
        f'("{name}" OR "{clean_key}") '
        f'urgences France '
        f'("fermeture temporaire" OR "grève" OR "SMUR" OR "communiqué de presse") '
        f'2022..2026'
    )

    try:
        # Search Official
        res_official = tavily_client.search(
            query=query_official, 
            search_depth="advanced", 
            max_results=10)
        
        # Search Unofficial
        # Advanced depth is crucial when searching open web to extract clean text from news articles
        res_unofficial = tavily_client.search(
            query=query_unofficial, 
            search_depth="advanced", 
            max_results=10 # Increased to catch press releases
        )

        # 3. Save
        entry = {
            "id": record_id,
            "finess": finess,
            "hospital_name": name,
            "official_data": res_official.get('results', []),
            "unofficial_data": res_unofficial.get('results', []),
            "retrieved_at": time.strftime("%Y-%m-%d %H:%M:%S")
        }

        with open(output_path, "a", encoding="utf-8") as f:
            f.write(json.dumps(entry) + "\n")
        
        print(f"✅ Deep Search Complete: {name}")
        time.sleep(1)

    except Exception as e:
        print(f"❌ Error with {name}: {str(e)}")
print(f"Deep context dataset saved in: {output_path}")

✅ Deep Search Complete: CH DE FLEYRIAT
✅ Deep Search Complete: CH BUGEY SUD
✅ Deep Search Complete: CH DU HAUT BUGEY - GEOVREISSET
✅ Deep Search Complete: CLINIQUE CONVERT
✅ Deep Search Complete: HOPITAL PRIVE D AMBERIEU
✅ Deep Search Complete: CH SAINT-QUENTIN
✅ Deep Search Complete: CH SAINT-QUENTIN
✅ Deep Search Complete: CH LAON
✅ Deep Search Complete: CH LAON
✅ Deep Search Complete: CH SOISSONS
✅ Deep Search Complete: CH SOISSONS
✅ Deep Search Complete: CH CHAUNY
✅ Deep Search Complete: CH CHATEAU-THIERRY
✅ Deep Search Complete: CH HIRSON
✅ Deep Search Complete: HOPITAL PRIVE SAINT CLAUDE
✅ Deep Search Complete: CH DE MOULINS
✅ Deep Search Complete: CH DE MONTLUCON
✅ Deep Search Complete: CH JACQUES LACARIN VICHY
✅ Deep Search Complete: CHI DE MANOSQUE LOUIS RAFFALLI
✅ Deep Search Complete: CHI DES ALPES DU SUD SITE DE SISTERON
✅ Deep Search Complete: CENTRE HOSPITALIER DE DIGNE LES BAINS
✅ Deep Search Complete: CH DES ESCARTONS DE BRIANCON
✅ Deep Search Complete: CENTRE HOSPITALI